# 📥 RAG Retrieval (Elasticsearch)

This notebook handles the **expensive** part of the RAG evaluation pipeline:  
connecting to Elasticsearch, running batch retrieval across strategies, and saving raw results to disk.

## Pipeline
1. **Load** the QA dataset (questions + ground-truth Wikipedia IDs)
2. **Connect** to the Elasticsearch index
3. **Retrieve** top-K documents for every question using each strategy
4. **Save** raw retrieval results as Parquet files

## Strategies
- **Dense Vector Search** (`approximation`): Semantic similarity via embeddings
- **BM25 Keyword Search** (`bm25`): Traditional full-text search
- **Hybrid Search** (`hybrid`): Combined vector + BM25

## Output
Results are saved to `{COLLECTION_ROOT}/{OUTPUT_NAME}/` as `results_{strategy}.parquet`.  
These files are consumed by `rag_evaluation.ipynb` for metric computation and visualization.

> **Note:** Run this notebook once (or when you change retrieval settings).  
> The evaluation notebook can be re-run cheaply on saved results.

## 1. Configuration

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv

from rag.elasticsearch_rag_service import ElasticsearchRagService
from rag.utils import IndexingConfig
from config import DATA_DIR


load_dotenv()

# Elasticsearch Connection
ES_URL = "http://localhost:9200"
ES_USER = os.getenv("ELASTICSEARCH_USERNAME") or None
ES_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD") or None
OUTPUT_NAME = "nq_200"

# Collection & Index
COLLECTION_NAME = "wiki_full"
COLLECTION_ROOT = Path(DATA_DIR) / COLLECTION_NAME
QUESTIONS_PATH = COLLECTION_ROOT / f"{OUTPUT_NAME}.parquet"

# Embedding Configuration (MUST match indexing settings exactly!)
EMBEDDING_MODEL = "intfloat/multilingual-e5-small"  # Match rag_indexing.ipynb!
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# Retrieval Configuration
STRATEGIES = ["approximation", "bm25"]  # Strategies to evaluate
TOP_K = 10  # Primary metric for reporting (e.g., Recall@10 for popularity decile analysis)
K_VALUES_DETAILED = [1,3,5,10]  # K values for investigation/curves (e.g., 1-20 to see full trend)
MAX_QUESTIONS = None  # Set to limit for testing (e.g., 100), None for all

# kNN Search Configuration
# num_candidates controls how many vectors HNSW explores during approximate search.
# Default in langchain-elasticsearch is 50, which is far too low for large indices.
# Higher = better recall but slower. Recommended: 500-10000 for million-scale indices.
NUM_CANDIDATES = 1000  # Override the pathologically low default of 50

# Output Configuration
OUTPUT_FOLDER = OUTPUT_NAME  # Folder name for storing all results
RESULTS_DIR = COLLECTION_ROOT / OUTPUT_FOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Config loaded")
print(f"  Index: {COLLECTION_NAME} @ {ES_URL}")
print(f"  Questions: {QUESTIONS_PATH}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  Top-K (Primary Metric): {TOP_K}")
print(f"  K-Values for Investigation: {list(K_VALUES_DETAILED)}")
print(f"  kNN num_candidates: {NUM_CANDIDATES}")
print(f"  Will retrieve: {max(TOP_K, max(K_VALUES_DETAILED))} documents per query")
print(f"  Results: {RESULTS_DIR}")

✓ Config loaded
  Index: wiki_full @ http://localhost:9200
  Questions: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_200.parquet
  Embedding Model: intfloat/multilingual-e5-small
  Top-K (Primary Metric): 10
  K-Values for Investigation: [1, 3, 5, 10]
  kNN num_candidates: 1000
  Will retrieve: 10 documents per query
  Results: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_200


## 2. Load Questions

Load the QA dataset containing questions, ground-truth Wikipedia IDs, popularity scores, and decile labels.

In [2]:
print("Loading questions...")
qa_df = pd.read_parquet(QUESTIONS_PATH)

qa_df = qa_df.dropna(subset=["question_text"])

# Normalize Wikipedia IDs to string format for matching
qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(str).str.strip()

# Limit for testing if specified
if MAX_QUESTIONS:
    qa_df = qa_df.sample(n=min(MAX_QUESTIONS, len(qa_df)), random_state=42)
    print(f"  Limited to {len(qa_df)} questions for testing")

print(f"✓ Loaded {len(qa_df):,} questions")
print(f"  Unique docs: {qa_df['wikipedia_id'].nunique():,}")
print(f"  Datasets: {qa_df['dataset'].value_counts().to_dict() if 'dataset' in qa_df.columns else 'N/A'}")

# Display sample
print("\nSample questions:")
display(qa_df.head(3))

Loading questions...
✓ Loaded 1,144 questions
  Unique docs: 1,089
  Datasets: {'natural_questions': 1144}

Sample questions:


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,is_synthetic,decile
0,5074106293586903195,who is the current leader of the official oppo...,"[Andrew Scheer, ]",711418,List of Leaders of the Official Opposition (Ca...,1.866667,5.801144e+06,natural_questions,False,0
1,8919552861517454808,who played corde in attack of the clones,[Verónica Segura],7232669,Verónica Segura,7.287037,5.671906e+06,natural_questions,False,0
2,-4404375150543368489,who is the education minister of pakistan 2018,"[Shafqat Mahmood, ]",17201259,Minister for Education (Pakistan),2.428571,5.716288e+06,natural_questions,False,0


## 3. Connect to Elasticsearch Index

Initialize the `ElasticsearchRagService` and connect to the pre-built index.  
The embedding model and chunking parameters **must** match what was used during indexing (`rag_indexing.ipynb`).

In [3]:
print("Connecting to Elasticsearch index...")

config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider="huggingface",
    embedding_model=EMBEDDING_MODEL,
    trust_remote_code=True,
    use_progress=False
)

# Initialize service (strategy doesn't matter for loading, we'll override at query time)
service = ElasticsearchRagService(
    config=config,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
    strategy="hybrid"
)

service.load_index(COLLECTION_NAME)


print(f"✓ Connected to Elasticsearch index: {COLLECTION_NAME}")

Connecting to Elasticsearch index...
✓ Connected to Elasticsearch index: wiki_full


## 4. Run Batch Retrieval

For each strategy, retrieve the top-K documents for every question.  
This is the most **time-consuming** step — results are cached to Parquet afterwards.

Each result row contains:
- `topk_ids`: Wikipedia IDs of retrieved documents
- `topk_scores`: Relevance scores
- `topk_popularities`: Popularity values of retrieved documents

In [4]:
print("Running retrieval for all strategies...\n")

results_by_strategy = {}

for strategy in STRATEGIES:
    print(f"{'='*60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'='*60}")
    
    # Only use num_candidates for vector/hybrid strategies (irrelevant for BM25)
    nc = NUM_CANDIDATES if strategy in ("approximation", "hybrid") else None
    
    # Run batch retrieval with this strategy
    all_results = service.batch_retrieve(
        questions=qa_df["question_text"].tolist(),
        top_k=max(TOP_K, max(K_VALUES_DETAILED)),  # Retrieve enough for both primary and investigation
        strategy=strategy,
        num_candidates=nc,
    )
    
    # Process results
    rows = []
    for idx, (question_row, retrieved_docs) in enumerate(zip(qa_df.itertuples(), all_results)):
        expected_id = str(question_row.wikipedia_id).strip()
        
        # Extract retrieved doc IDs
        retrieved_ids = []
        retrieved_scores = []
        retrieved_popularities = []
        
        for doc, score in retrieved_docs:
            raw_id = doc.metadata.get("wikipedia_id", doc.metadata.get("id", ""))
            doc_id = str(int(float(raw_id))) if raw_id not in (None, "") else ""
            retrieved_ids.append(doc_id)
            retrieved_scores.append(score)
            retrieved_popularities.append(doc.metadata.get("popularity_avg"))
        
        rows.append({
            "question": question_row.question_text,
            "wikipedia_id": expected_id,
            "wikipedia_title": getattr(question_row, "wikipedia_title", None),
            "decile": getattr(question_row, "decile", -1),
            "popularity_avg": getattr(question_row, "popularity_avg", None),
            "dataset": getattr(question_row, "dataset", None),
            "topk_ids": retrieved_ids,
            "topk_scores": retrieved_scores,
            "topk_popularities": retrieved_popularities,
        })
    
    results_df = pd.DataFrame(rows)
    # Add pop_decile column immediately for later use
    results_df['pop_decile'] = results_df['decile']
    results_by_strategy[strategy] = results_df
    
    print(f"✓ Retrieved {len(results_df)} queries\n")

# Update STRATEGIES list to include all tested configurations
ALL_STRATEGIES = list(results_by_strategy.keys())
print(f"✅ All retrieval complete! Tested {len(ALL_STRATEGIES)} configurations:")
print(f"   {', '.join(ALL_STRATEGIES)}")

Running retrieval for all strategies...

Strategy: APPROXIMATION


Retrieving (approximation):  70%|███████   | 801/1144 [23:27<15:01,  2.63s/q]  

ConnectionTimeout: Connection timed out

## 5. Save Raw Results

Save retrieval results as Parquet files — one per strategy.  
These are the input for `rag_evaluation.ipynb`.

In [ ]:
print("Saving retrieval results...\n")

for strategy in ALL_STRATEGIES:
    results_df = results_by_strategy[strategy]
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    results_df.to_parquet(output_path)
    print(f"  ✓ Saved {strategy}: {output_path} ({len(results_df):,} rows)")

print(f"\n✅ All results saved to: {RESULTS_DIR}")
print(f"\nNext step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.")

Saving retrieval results...

  ✓ Saved approximation: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_500/results_approximation.parquet (2,227 rows)
  ✓ Saved bm25: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_500/results_bm25.parquet (2,227 rows)

✅ All results saved to: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/nq_500

Next step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.


## 6. Generate Metadata

Calculate decile boundaries (both unweighted and chunk-weighted) and save configuration metadata.  
This metadata is used for analysis and reproducibility.

In [ ]:
import json
import sys
from pathlib import Path

# Import the boundary calculation function from prepare_qa_dataset
sys.path.insert(0, str(Path.cwd() / "scripts"))
from prepare_qa_dataset import calculate_corpus_decile_boundaries

print("Calculating decile boundaries from corpus...\n")

# Locate corpus file
CORPUS_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"

if not CORPUS_PATH.exists():
    print(f"⚠️  Corpus not found at {CORPUS_PATH}")
    print("   Metadata will be saved without decile boundaries")
    metadata = {
        "collection_name": COLLECTION_NAME,
        "output_name": OUTPUT_NAME,
        "embedding_model": EMBEDDING_MODEL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "strategies": ALL_STRATEGIES,
        "top_k": TOP_K,
        "k_values_detailed": K_VALUES_DETAILED,
        "num_candidates": NUM_CANDIDATES,
        "num_questions": len(qa_df),
        "corpus_path": str(CORPUS_PATH),
        "corpus_found": False,
    }
else:
    print(f"Reading corpus: {CORPUS_PATH}")
    
    # Use the shared function from prepare_qa_dataset.py
    boundaries_unweighted, boundaries_weighted, stats, _ = calculate_corpus_decile_boundaries(
        corpus_path=CORPUS_PATH,
        batch_size=100_000,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    
    print(f"\n✓ Calculated decile boundaries")
    print(f"  Unique documents: {stats['unique_documents_with_popularity']:,}")
    print(f"  Total chunks: {stats['total_chunks_after_splitting']:,}")
    
    # Build metadata
    metadata = {
        "collection_name": COLLECTION_NAME,
        "output_name": OUTPUT_NAME,
        "embedding_model": EMBEDDING_MODEL,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "strategies": ALL_STRATEGIES,
        "top_k": TOP_K,
        "k_values_detailed": K_VALUES_DETAILED,
        "num_candidates": NUM_CANDIDATES,
        "num_questions": len(qa_df),
        "corpus_path": str(CORPUS_PATH),
        "corpus_found": True,
        "corpus_stats": stats,
        "decile_boundaries_unweighted": boundaries_unweighted.tolist(),
        "decile_boundaries_chunk_weighted": boundaries_weighted.tolist(),
        "decile_boundaries_chunk_weighted_config": {
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
        },
    }

# Save metadata
metadata_path = RESULTS_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Metadata saved to: {metadata_path}")
print(f"\nReady for evaluation! Open rag_evaluation.ipynb to analyze results.")


INFO - Calculating decile boundaries from 5,903,530 corpus documents...


Calculating decile boundaries from corpus...

Reading corpus: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full/wiki_corpus.parquet


Reading corpus:   0%|          | 0/60 [00:00<?, ?it/s]

INFO - Processed 5,903,530 rows → 5,890,044 unique docs with popularity
INFO - Total chunks (after splitting): 24,651,978
INFO - Decile boundaries (unweighted - 1 doc = 1 count):
INFO -   Decile 0: [1.0000, 8.1875)
INFO -   Decile 1: [8.1875, 15.1250)
INFO -   Decile 2: [15.1250, 25.3750)
INFO -   Decile 3: [25.3750, 41.7708)
INFO -   Decile 4: [41.7708, 69.6042)
INFO -   Decile 5: [69.6042, 120.2708)
INFO -   Decile 6: [120.2708, 224.0833)
INFO -   Decile 7: [224.0833, 480.1458)
INFO -   Decile 8: [480.1458, 1437.6666)
INFO -   Decile 9: [1437.6666, 175763168.0000)
INFO - Decile boundaries (chunk-weighted - chunk_size=1000, overlap=100):
INFO -   Decile 0: [1.0000, 23.6667)
INFO -   Decile 1: [23.6667, 55.7292)
INFO -   Decile 2: [55.7292, 112.7083)
INFO -   Decile 3: [112.7083, 217.6667)
INFO -   Decile 4: [217.6667, 415.1250)
INFO -   Decile 5: [415.1250, 810.4583)
INFO -   Decile 6: [810.4583, 1674.5416)
INFO -   Decile 7: [1674.5416, 3895.0000)
INFO -   Decile 8: [3895.0000, 11655


✓ Calculated decile boundaries
  Unique documents: 5,890,044
  Total chunks: 24,651,978


NameError: name 'ALL_STRATEGIES' is not defined